# Libraries

In [1]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()

project_root = current_dir.parents[2]

csv_path_subject = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Participant_Status_02Mar2026.csv"
csv_path_demo = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Demographics_02Mar2026.csv"
csv_path_econ = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Socio-Economics_02Mar2026.csv"
csv_path_age = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Age_at_visit_02Mar2026.csv"
csv_path_vital = project_root / "DATA_PPMI" / "Subject_Characteristics" / "Vital_Signs_02Mar2026.csv"


# Subject Data

In [2]:
subject_data=pd.read_csv(csv_path_subject)
subject_data=subject_data.loc[subject_data['COHORT_DEFINITION'].isin(["Parkinson's Disease", "Prodromal"])]
patnos=subject_data['PATNO'].unique().tolist()
print("Subject data:",subject_data.shape)
cols_imp=['PATNO','ENRLLRRK2','ENRLGBA','COHORT_DEFINITION']
subject_data=subject_data[cols_imp].copy()
print('Subject data is nan:',subject_data.isna().sum().sum())
subject_data.head() # no hay nan

Subject data: (7697, 30)
Subject data is nan: 0


,PATNO,ENRLLRRK2,ENRLGBA,COHORT_DEFINITION
1,3001,0,0,Parkinson's Disease
2,3002,0,0,Parkinson's Disease
3,3003,0,0,Parkinson's Disease
5,3005,0,0,Parkinson's Disease
6,3006,0,0,Parkinson's Disease


# Demographic Data

In [3]:
demo_data=pd.read_csv(csv_path_demo)
demo_data=demo_data.loc[demo_data['PATNO'].isin(patnos)]
print("Demographics data:",demo_data.shape)

demo_data=demo_data[['PATNO','SEX','RAWHITE']].copy() # no hay nan
demo_data.fillna(demo_data.mean(numeric_only=True), inplace=True)
print(demo_data.isna().sum().sum()) # no hay nan
demo_data.head()

Demographics data: (7621, 29)
0


,PATNO,SEX,RAWHITE
1,3001,1.0,1.0
2,3002,0.0,1.0
3,3003,0.0,1.0
5,3005,0.0,1.0
6,3006,0.0,1.0


# Socio-Economics Data

In [4]:
socio_data=pd.read_csv(csv_path_econ)
socio_data=socio_data.loc[(socio_data['PATNO'].isin(patnos))&(socio_data['EVENT_ID'].isin(['BL','SC']))]
print("Socio-Economics data:",socio_data.shape)
socio_data=socio_data[['PATNO','EDUCYRS']].copy()
print('Socio-Economics data is nan:',socio_data.isna().sum().sum())
socio_data.fillna(socio_data.mean(numeric_only=True), inplace=True)
socio_data.head() # no hay nan

Socio-Economics data: (6648, 11)
Socio-Economics data is nan: 10


,PATNO,EDUCYRS
2,3001,16.000000
4,3002,16.000000
6,3003,16.000000
10,3005,16.353721
11,3006,14.000000


# Subject/Demo/SocioEcon

In [5]:
# Merge datasets
data = pd.merge(subject_data, demo_data, on='PATNO', how='outer')
data = pd.merge(data, socio_data, on='PATNO', how='outer')
print("Merged data:", data.shape)
data.head()

Merged data: (7697, 7)


,PATNO,ENRLLRRK2,ENRLGBA,COHORT_DEFINITION,SEX,RAWHITE,EDUCYRS
0,3001,0,0,Parkinson's Disease,1.0,1.0,16.000000
1,3002,0,0,Parkinson's Disease,0.0,1.0,16.000000
2,3003,0,0,Parkinson's Disease,0.0,1.0,16.000000
3,3005,0,0,Parkinson's Disease,0.0,1.0,16.353721
4,3006,0,0,Parkinson's Disease,0.0,1.0,14.000000


# Age at Visit + Data Total

In [6]:
import pandas as pd
import numpy as np

AGE_COL = 'AGE_AT_VISIT'
ordered_visits = ['BL', 'V04', 'V06', 'V08', 'V10', 'V12']
visit_offsets = {'BL': 0, 'V04': 1, 'V06': 2, 'V08': 3, 'V10': 4, 'V12': 5}

# --- 1) Carga + filtro
age_data = pd.read_csv(csv_path_age)
age_data = age_data.loc[
    age_data['PATNO'].isin(patnos) &
    age_data['EVENT_ID'].isin(['BL', 'SC'] + ordered_visits[1:])   # SC + visitas
].copy()

# --- 2) Si hay BL y SC, elimina SC
patnos_bl = set(age_data.loc[age_data['EVENT_ID'] == 'BL', 'PATNO'])
patnos_sc = set(age_data.loc[age_data['EVENT_ID'] == 'SC', 'PATNO'])
patnos_con_bl_y_sc = patnos_bl.intersection(patnos_sc)

age_data = age_data[
    ~((age_data['PATNO'].isin(patnos_con_bl_y_sc)) & (age_data['EVENT_ID'] == 'SC'))
].copy()

# --- 3) SC -> BL (si quedó alguno)
age_data['EVENT_ID'] = age_data['EVENT_ID'].replace({'SC': 'BL'})

# --- 4) Nos quedamos con columnas necesarias y sin duplicados por PATNO+EVENT
age_data = age_data[['PATNO', 'EVENT_ID', AGE_COL]].drop_duplicates(subset=['PATNO', 'EVENT_ID']).copy()

# --- 5) Calcular/asegurar edad baseline por PATNO
# Si existe BL, usarla. Si no, derivarla desde la visita con menor offset disponible.
age_data['offset'] = age_data['EVENT_ID'].map(visit_offsets)

# baseline directo si existe
bl_from_bl = (
    age_data.loc[age_data['EVENT_ID'] == 'BL', ['PATNO', AGE_COL]]
    .set_index('PATNO')[AGE_COL]
)

# baseline derivado: AGE_BL = AGE_VISIT - offset(visit) usando la visita con menor offset
idx_min_offset = age_data.groupby('PATNO')['offset'].idxmin()
bl_from_any = (
    age_data.loc[idx_min_offset, ['PATNO', AGE_COL, 'offset']]
    .assign(bl_age=lambda d: d[AGE_COL].astype(float) - d['offset'].astype(float))
    .set_index('PATNO')['bl_age']
)

# baseline final: BL si existe, si no el derivado
bl_age = bl_from_any.copy()
bl_age.update(bl_from_bl)

# --- 6) Generar tabla completa PATNO x visitas (todas)
all_patnos = age_data['PATNO'].unique()
grid = pd.MultiIndex.from_product([all_patnos, ordered_visits], names=['PATNO', 'EVENT_ID']).to_frame(index=False)
grid['offset'] = grid['EVENT_ID'].map(visit_offsets)

# edad imputada por regla (BL + offset)
grid[AGE_COL] = grid['PATNO'].map(bl_age).astype(float) + grid['offset'].astype(float)

# --- 7) Mantener edades originales cuando existan (no pisarlas)
orig = age_data[['PATNO', 'EVENT_ID', AGE_COL]].copy()
full_age = grid.merge(orig, on=['PATNO', 'EVENT_ID'], how='left', suffixes=('_imputed', '_orig'))

full_age[AGE_COL] = full_age[f'{AGE_COL}_orig'].combine_first(full_age[f'{AGE_COL}_imputed'])
full_age = full_age[['PATNO', 'EVENT_ID', AGE_COL]].sort_values(['PATNO', 'EVENT_ID']).reset_index(drop=True)

# --- 8) Merge con tu data
# OJO: si data es 1 fila por PATNO, esto duplicará a 6 filas por PATNO.
# Si quieres mantener data a nivel visita, data también debe tener EVENT_ID.
data = pd.merge(data, full_age, on='PATNO', how='inner')

print("Merged data with age:", data.shape)
data.head()

Merged data with age: (44418, 9)


,PATNO,ENRLLRRK2,ENRLGBA,COHORT_DEFINITION,SEX,RAWHITE,EDUCYRS,EVENT_ID,AGE_AT_VISIT
0,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,BL,65.1
1,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V04,66.2
2,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V06,67.3
3,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V08,68.3
4,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V10,69.2


# Final DF Result

In [7]:
print("Final merged data:", data.shape)
print('Final Cols:', data.columns.tolist())
print("Missing values:\n", data.isna().sum().sum())
csv_path_data = project_root / "DATA_PPMI" / "Results" / "SUBJECT_CHARACTERISTICS.csv"
data.to_csv(csv_path_data, index=False)
data.head()


Final merged data: (44418, 9)
Final Cols: ['PATNO', 'ENRLLRRK2', 'ENRLGBA', 'COHORT_DEFINITION', 'SEX', 'RAWHITE', 'EDUCYRS', 'EVENT_ID', 'AGE_AT_VISIT']
Missing values:
 5496


,PATNO,ENRLLRRK2,ENRLGBA,COHORT_DEFINITION,SEX,RAWHITE,EDUCYRS,EVENT_ID,AGE_AT_VISIT
0,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,BL,65.1
1,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V04,66.2
2,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V06,67.3
3,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V08,68.3
4,3001,0,0,Parkinson's Disease,1.0,1.0,16.0,V10,69.2


In [8]:
data['EVENT_ID'].value_counts()

EVENT_ID
BL     7403
V04    7403
V06    7403
V08    7403
V10    7403
V12    7403
Name: count, dtype: int64